In [ ]:
#Install requirements
%pip install -r "../requirements.txt"

In [ ]:
#Import required libraries
import cv2
import numpy as np
from pathlib import Path
import pandas as pd
import os
import sys

In [ ]:
#Function to calculate Tissue Area from the original RGB images before color deconvolution
def create_tissue_registry(root_dir, marker_name, regions, groups, tissue_threshold=240):
    registry_data = []
    #Scans original RGB images to calculate total tissue area.
    for region in regions:
        for group in groups:
            # Path to original RGB images: root/MARKER/REGION/GROUP/SUBJECT/
            base_path = Path(root_dir) / marker_name / region / group
            
            if not base_path.exists():
                print(f"Skipping missing path: {base_path}")
                continue
                
            # Find subject folders
            subjects = [s for s in base_path.iterdir() if s.is_dir()]
            
            for sub_path in subjects:
                # Find all image files in the subject folder
                image_files = list(sub_path.glob('*.jpg'))
                
                print(f"Processing {region}/{group}/{sub_path.name} ({len(image_files)} images)")
                
                for img_file in image_files:
                    img = cv2.imread(str(img_file), cv2.IMREAD_GRAYSCALE)
                    if img is not None:
                        tissue_pixels = np.sum(img < tissue_threshold)
                        
                        registry_data.append({
                            'Image_Name': img_file.name,
                            'Subject_ID': sub_path.name,
                            'Region': region,
                            'Group': group,
                            'Total_Tissue_Pixels': int(tissue_pixels)
                        })

    # Save to CSV
    df_registry = pd.DataFrame(registry_data)
    output_path = Path(root_dir) / marker_name / f"{marker_name}_tissue_registry.csv"
    df_registry.to_csv(output_path, index=False)
    
    print(f"\n Tissue Registry created at: {output_path}")
    return df_registry

In [ ]:
#RUN CONFIGURATION (adjust the root_path as needed)
ROOT = r'root_path'
MARKER = ['AQP4', 'GFAP'] #adjust as needed
REGIONS = ['GM', 'WM'] #adjust as needed
GROUPS = ['Controls', 'Patients'] #adjust as needed

if __name__ == "__main__":
    create_tissue_registry(ROOT, MARKER, REGIONS, GROUPS)